In [1]:
import pandas as pd
import xarray as xr
import os


In [3]:
output_dir = "./output"


In [12]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df_temp = pd.read_csv("./dataverse_files/Temperature profiles/temperature.csv")
df_meta = pd.read_csv("./dataverse_files/meta.csv")
df_temp.columns = [c.strip() for c in df_temp.columns]
df_meta['Borehole ID'] = df_meta['Borehole ID'].str.strip()

df_melted = df_temp.melt(
    id_vars=['depth [m]'], 
    var_name='borehole', 
    value_name='temperature')

df_melted = df_melted.rename(columns={'depth [m]': 'depth'})
ds_temp = df_melted.set_index(['depth', 'borehole']).to_xarray()

df_meta_indexed = df_meta.set_index('Borehole ID')

ds_meta = df_meta_indexed.to_xarray()
ds_meta = ds_meta.rename({'Borehole ID': 'borehole'})

ds_final = xr.merge([ds_temp, ds_meta])

metadata_columns = [
    'Place name', 'Geographic location', 'Site type', 'Data Source', 
    'Data DOI', 'Science Source', 'Science DOI', 'Date', 
    'Longitude [°E]', 'Latitude [°N]', 'Location Source', 
    'Depth of top measurement [m]', 'Depth of bottom measurement [m]', 
    'Ice thickness [m]', 'Coverage [% of thickness]', 
    'Ice thickness source', 'Note']

ds_final = ds_final.set_coords(metadata_columns)

ds_final.temperature.attrs = {'units': 'Celsius', 'standard_name': 'ice_temperature'}
ds_final.depth.attrs = {'units': 'm', 'standard_name': 'depth'}

In [14]:
output_path = os.path.join(output_dir, 'temp_profiles.nc')
ds_final.to_netcdf(output_path)


/tmp/ipykernel_2135327/3439781361.py:2: SerializationWarning: coordinate 'Data Source' has a space in its name, which means it cannot be marked as a coordinate on disk and will be saved as a data variable instead
  ds_final.to_netcdf(output_path)
/tmp/ipykernel_2135327/3439781361.py:2: SerializationWarning: coordinate 'Science Source' has a space in its name, which means it cannot be marked as a coordinate on disk and will be saved as a data variable instead
  ds_final.to_netcdf(output_path)
/tmp/ipykernel_2135327/3439781361.py:2: SerializationWarning: coordinate 'Ice thickness source' has a space in its name, which means it cannot be marked as a coordinate on disk and will be saved as a data variable instead
  ds_final.to_netcdf(output_path)
/tmp/ipykernel_2135327/3439781361.py:2: SerializationWarning: coordinate 'Latitude [°N]' has a space in its name, which means it cannot be marked as a coordinate on disk and will be saved as a data variable instead
  ds_final.to_netcdf(output_path